#Práctica con MLxtend: Reglas de asociación.


##Elección del Dataset

a) El dataset seleccionado contiene más de 10.000 URLs, cada una descrita por 31 características que permiten identificar posibles sitios web de phishing. Este conjunto de datos proviene de OpenML [dataset 4534](https://www.openml.org/search?type=data&status=active&id=4534) y está diseñado para analizar atributos relacionados con la legitimidad de los sitios web.
Para cada característica, las URLs están clasificadas con valores de 1, 0 o -1, indicando si son phishing, dudosa o legitima, respectivamente. Esto permite aplicar técnicas de minería de datos y aprendizaje automático para detectar patrones asociados a sitios potencialmente fraudulentos.

Dejamos a disposición la definición particular de cada característica analizada en el Dataset (https://archive.ics.uci.edu/ml/machine-learning-databases/00327/Phishing Websites Features.docx) junto con la [UCI fuente](https://archive.ics.uci.edu/dataset/327/phishing+websites)

##Objetivo

Aplicar reglas de asociación en el dataset de las cuales esperamos encontrar patrones frecuentes dentro de las URLs maliciosas. Se podrían encontrar reglas como: "Si la longitud de la URL es muy larga (característica A = 1) y el uso de símbolos sospechosos en la URL es alto (característica B = 1), entonces es probable que este desactivado el click derecho sobre el adress bar (característica C = 1).

**Objetivo:** Identificar reglas de asociación en el dataset de características de URLs maliciosas, que nos permitan encontrar combinaciones sospechosas. Estas reglas podrían ser utilizadas para generar listas de bloqueo y ser integradas en herramientas como extensiones de navegador para prevenir phishing.

##Pre-procesamiento del Dataset


1.   Se carga la librería que permite *acceder al dataset*.
2.   Se obtiene un dataset para el *pre-procesamiento de datos*.
3.   Se crea un *dataframe* donde se separa la información almacenada y la concatenamos.
4.   Se filtran las columnas donde las URL son *clasificadas como pishing* para buscar reglas únicamente de las *URL maliciosas* dentro del dataset.
5.   Se borra la columna 'Result', ya que luego de filtrar las filas, ya no contiene información  útil.
6. Con la libreria *pandas* se transforma el dataframe a un formato *one hot encode*.



In [ ]:
# soporte para cargar dataset de https://www.openml.org/
!pip install openml
import openml
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.0/158.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 34.8 MB/s eta 0:00:00
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=b23f81c580810c69427d29890a54154d9b89a6ca3eaf3f974781b093a6d881d5
  Stored in directory: /root/.cache/pip/wheels/5d/2a/9c/3895d9617f8f49a0883ba686326d598e78a1c2f54fe3cae86d
Successfully built liac-arff


In [ ]:
# indicamos cual dataset queremos utilizar, en este caso el nro. 24
dataset = openml.datasets.get_dataset(4534)

# separamos las información almacenada en el dataset
X, y, categorical_indicator, attribute_names = dataset.get_data(
    dataset_format='dataframe',
    target=dataset.default_target_attribute
)

#  concatenamos la información relevante en un único DataFrame
df = pd.concat([X, y], axis=1)

#
# filtramos las filas que solo sean phishing(Result == 1).
df_phishing = df[df['Result'] == '1']

# eliminamos la columna Result, ya que no aporta información por ser todas las filas True.
df_phishing = df_phishing.drop('Result', axis=1)

df_phishing


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,RightClick,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report
4,1,0,-1,1,1,-1,1,1,-1,1,...,1,-1,1,-1,-1,0,-1,1,1,1
5,-1,0,-1,1,-1,-1,1,1,-1,1,...,1,1,1,1,1,1,-1,1,-1,-1
8,1,0,-1,1,1,-1,1,1,-1,1,...,1,1,1,1,-1,1,1,1,0,1
10,1,1,1,1,1,-1,0,1,1,1,...,1,1,1,-1,1,1,1,1,-1,-1
14,1,1,-1,1,1,1,-1,1,-1,1,...,1,1,1,1,-1,1,-1,1,-1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11044,-1,-1,-1,1,-1,-1,1,-1,-1,1,...,1,1,1,1,-1,0,-1,1,1,1
11045,1,-1,1,1,1,-1,1,-1,-1,1,...,1,1,1,1,1,0,-1,1,0,1
11046,-1,-1,1,1,1,-1,1,1,-1,1,...,1,1,1,1,1,0,-1,1,1,1
11048,1,-1,1,1,1,-1,-1,1,1,1,...,1,1,1,1,1,0,-1,1,0,1


In [ ]:
# Utilizamos la libreria Pandas para conseguir el dataset en formato "one-hot encoded"
df_dummies = pd.get_dummies(df_phishing)
df_dummies

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,having_IP_Address_-1,having_IP_Address_1,URL_Length_1,URL_Length_0,URL_Length_-1,Shortining_Service_1,Shortining_Service_-1,having_At_Symbol_1,having_At_Symbol_-1,double_slash_redirecting_-1,...,web_traffic_1,Page_Rank_-1,Page_Rank_1,Google_Index_1,Google_Index_-1,Links_pointing_to_page_1,Links_pointing_to_page_0,Links_pointing_to_page_-1,Statistical_report_-1,Statistical_report_1
4,False,True,False,True,False,False,True,True,False,False,...,False,True,False,True,False,True,False,False,False,True
5,True,False,False,True,False,False,True,True,False,True,...,True,True,False,True,False,False,False,True,True,False
8,False,True,False,True,False,False,True,True,False,False,...,True,False,True,True,False,False,True,False,False,True
10,False,True,True,False,False,True,False,True,False,False,...,True,False,True,True,False,False,False,True,True,False
14,False,True,True,False,False,False,True,True,False,False,...,True,True,False,True,False,False,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11044,True,False,False,False,True,False,True,True,False,True,...,False,True,False,True,False,True,False,False,False,True
11045,False,True,False,False,True,True,False,True,False,False,...,False,True,False,True,False,False,True,False,False,True
11046,True,False,False,False,True,True,False,True,False,False,...,False,True,False,True,False,True,False,False,False,True
11048,False,True,False,False,True,True,False,True,False,False,...,False,True,False,True,False,False,True,False,False,True


## Reglas de asociación.

1. Se calcula los itemsets más frecuentes.
2. Se hace un filtrado sobre los itemsets para estudiar con detalle los atributos relacionados.
3. Se generan las reglas de asociación.



In [ ]:
# Calculamo los itemsset más frecuentes
itemsets=apriori(df_dummies, min_support=0.8, use_colnames=True)
itemsets['length'] = itemsets['itemsets'].apply(lambda x: len(x))
itemsets


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,support,itemsets,length
0,0.848952,(Shortining_Service_1),1
1,0.867143,(having_At_Symbol_1),1
2,0.859185,(double_slash_redirecting_1),1
3,0.914406,(SSLfinal_State_1),1
4,0.814195,(Favicon_1),1
...,...,...,...
115,0.801364,"(on_mouseover_1, Iframe_1, Favicon_1, popUpWid...",5
116,0.802826,"(Iframe_1, Favicon_1, RightClick_1, popUpWidno...",5
117,0.801364,"(on_mouseover_1, Iframe_1, Favicon_1, RightCli...",5
118,0.801364,"(on_mouseover_1, Iframe_1, RightClick_1, popUp...",5


In [ ]:
# Filtramos los itemset mas frecuentes según diferentes métricas
itemsets[ (itemsets['support']>0.8)&
                   (itemsets['length']>=4)]
itemsets

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,support,itemsets,length
0,0.848952,(Shortining_Service_1),1
1,0.867143,(having_At_Symbol_1),1
2,0.859185,(double_slash_redirecting_1),1
3,0.914406,(SSLfinal_State_1),1
4,0.814195,(Favicon_1),1
...,...,...,...
115,0.801364,"(on_mouseover_1, Iframe_1, Favicon_1, popUpWid...",5
116,0.802826,"(Iframe_1, Favicon_1, RightClick_1, popUpWidno...",5
117,0.801364,"(on_mouseover_1, Iframe_1, Favicon_1, RightCli...",5
118,0.801364,"(on_mouseover_1, Iframe_1, RightClick_1, popUp...",5


In [ ]:
# Creamos las reglas de asociación
rules = association_rules(itemsets, metric="confidence", min_threshold=0.9)
rules

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(Shortining_Service_1),(double_slash_redirecting_1),0.848952,0.859185,0.838070,0.987182,1.148975,0.108664,10.985698,0.858400
1,(double_slash_redirecting_1),(Shortining_Service_1),0.859185,0.848952,0.838070,0.975425,1.148975,0.108664,6.146475,0.920775
2,(Shortining_Service_1),(HTTPS_token_1),0.848952,0.824427,0.808511,0.952363,1.155181,0.108611,3.685613,0.889353
3,(HTTPS_token_1),(Shortining_Service_1),0.824427,0.848952,0.808511,0.980693,1.155181,0.108611,7.823649,0.765123
4,(Shortining_Service_1),(Abnormal_URL_1),0.848952,0.833523,0.807211,0.950832,1.140739,0.099590,3.385900,0.816799
...,...,...,...,...,...,...,...,...,...,...
639,"(RightClick_1, port_1)","(on_mouseover_1, Favicon_1, popUpWidnow_1, Ifr...",0.870229,0.801364,0.801364,0.920866,1.149123,0.103994,2.510118,1.000000
640,"(popUpWidnow_1, port_1)","(on_mouseover_1, Favicon_1, RightClick_1, Ifra...",0.802826,0.807536,0.801364,0.998179,1.236080,0.153053,105.705430,0.968641
641,(Favicon_1),"(on_mouseover_1, Iframe_1, RightClick_1, popUp...",0.814195,0.801364,0.801364,0.984241,1.228207,0.148897,12.604567,1.000000
642,(popUpWidnow_1),"(on_mouseover_1, Iframe_1, Favicon_1, RightCli...",0.806724,0.807536,0.801364,0.993356,1.230107,0.149905,28.968727,0.967854


In [ ]:
# Agregamos la longitud del antecedente y del consecuente
rules['antecedents_length']=rules['antecedents'].apply(lambda x: len(x))
rules['consequents_length']=rules['consequents'].apply(lambda x: len(x))
rules


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedents_length,consequents_length
0,(Shortining_Service_1),(double_slash_redirecting_1),0.848952,0.859185,0.838070,0.987182,1.148975,0.108664,10.985698,0.858400,1,1
1,(double_slash_redirecting_1),(Shortining_Service_1),0.859185,0.848952,0.838070,0.975425,1.148975,0.108664,6.146475,0.920775,1,1
2,(Shortining_Service_1),(HTTPS_token_1),0.848952,0.824427,0.808511,0.952363,1.155181,0.108611,3.685613,0.889353,1,1
3,(HTTPS_token_1),(Shortining_Service_1),0.824427,0.848952,0.808511,0.980693,1.155181,0.108611,7.823649,0.765123,1,1
4,(Shortining_Service_1),(Abnormal_URL_1),0.848952,0.833523,0.807211,0.950832,1.140739,0.099590,3.385900,0.816799,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
639,"(RightClick_1, port_1)","(on_mouseover_1, Favicon_1, popUpWidnow_1, Ifr...",0.870229,0.801364,0.801364,0.920866,1.149123,0.103994,2.510118,1.000000,2,4
640,"(popUpWidnow_1, port_1)","(on_mouseover_1, Favicon_1, RightClick_1, Ifra...",0.802826,0.807536,0.801364,0.998179,1.236080,0.153053,105.705430,0.968641,2,4
641,(Favicon_1),"(on_mouseover_1, Iframe_1, RightClick_1, popUp...",0.814195,0.801364,0.801364,0.984241,1.228207,0.148897,12.604567,1.000000,1,5
642,(popUpWidnow_1),"(on_mouseover_1, Iframe_1, Favicon_1, RightCli...",0.806724,0.807536,0.801364,0.993356,1.230107,0.149905,28.968727,0.967854,1,5


### Filtrado por atributos.

Un valor alto de convicción significa que el consecuente depende en gran medida del antecedente. Por ejemplo, en el caso de una puntuación de confianza perfecta, el denominador se convierte en 0 (debido a 1 - 1) para el cual la puntuación de convicción se define como "inf". De manera similar a la elevación, si los elementos son independientes, la convicción es 1.

La columna lift indica cuánto más probable es que dos ítems ocurran juntos en comparación con si fueran completamente independientes.

Por otra parte, cuando la columna zhangs_metric tiene un valor de uno indica que no hay relación significativa entre el antecedente y el consecuente; su ocurrencia conjunta es lo que se esperaría si fueran independientes.


In [ ]:
# Filtramos las reglas por diferentes atributos
rules[(rules['conviction']>26) &
      (rules['lift']>1.23) &
      (rules['zhangs_metric']!=1)&
      (rules['antecedents_length']>1)]


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric,antecedents_length,consequents_length
254,"(popUpWidnow_1, port_1)","(on_mouseover_1, Favicon_1)",0.802826,0.808835,0.801364,0.998179,1.234094,0.152010,104.991807,0.962040,2,2
280,"(popUpWidnow_1, port_1)","(Favicon_1, RightClick_1)",0.802826,0.812896,0.802826,1.000000,1.230170,0.150212,inf,0.948929,2,2
302,"(popUpWidnow_1, port_1)","(Favicon_1, Iframe_1)",0.802826,0.811597,0.802826,1.000000,1.232139,0.151255,inf,0.955519,2,2
434,"(on_mouseover_1, popUpWidnow_1, port_1)","(Favicon_1, RightClick_1)",0.801364,0.812896,0.801364,1.000000,1.230170,0.149939,inf,0.941946,3,2
438,"(RightClick_1, popUpWidnow_1, port_1)","(on_mouseover_1, Favicon_1)",0.802826,0.808835,0.801364,0.998179,1.234094,0.152010,104.991807,0.962040,3,2
448,"(popUpWidnow_1, port_1)","(on_mouseover_1, Favicon_1, RightClick_1)",0.802826,0.808835,0.801364,0.998179,1.234094,0.152010,104.991807,0.962040,2,3
489,"(on_mouseover_1, popUpWidnow_1, port_1)","(Favicon_1, Iframe_1)",0.801364,0.811597,0.801364,1.000000,1.232139,0.150980,inf,0.948487,3,2
492,"(port_1, popUpWidnow_1, Iframe_1)","(on_mouseover_1, Favicon_1)",0.802826,0.808835,0.801364,0.998179,1.234094,0.152010,104.991807,0.962040,3,2
503,"(popUpWidnow_1, port_1)","(on_mouseover_1, Favicon_1, Iframe_1)",0.802826,0.807536,0.801364,0.998179,1.236080,0.153053,105.705430,0.968641,2,3
517,"(port_1, popUpWidnow_1, Iframe_1)","(Favicon_1, RightClick_1)",0.802826,0.812896,0.802826,1.000000,1.230170,0.150212,inf,0.948929,3,2


Las *reglas* que se han encontrado contienen los siguientes *Itemsets*:

* **Favicon**: Un icono asociado a una página web específica. Si un Favicon está cargado desde un dominio externo al que se muestra en la barra de búsqueda es phishing.

* **Right click**: Phishers usa JavaScript para deshabilitar la función de click derecho con el fin de que los usuarios no puedan ver y guardar el código de la página.

* **Port**: No usa los puertos legitimos de https.

* **Mouse Over**: Cuando el mouse está sobre el link si es phishing se detecta y deshabilita funciones como right click y status bar.

* **Iframe**: Los atacantes pueden superponer una página maliciosa sobre la página legítima sin que el usuario lo note. A su vez, se puede dar el caso de que sobre un botón o enlace se superpone un iframe invisible que puede ser malicioso.Si el Iframe es invisible nos indica que es phishing.

* **Using Pop-up Window**: No es habitual encontrar un sitio web legítimo que solicite a los usuarios que envíen su información personal a través de una ventana emergente.

## Conclusión.

Una vez que se generaron las reglas observamos que las mismas tienen muchas correlación entre el concecuente y el antecedente, lo que nos permitio hacer un filtrado por atrubutos (lift, conviction, zhangs metric y antecedents length) con valores significativos y poder analizar los resultados.

Los patrones más comunes entre enlaces que son phishing nos indican qué características co-existen comunmente en una URL maliciosa.

Las reglas encontradas no solo pueden utilizarse para el objetivo planteado, sino también para mejorar un modelo de clasificación proporcionando información más valiosas de las características de cada URL pudiendo obtener una precisión mucho mayor.